In [61]:
import pandas as pd
import pyspark
from pyspark.sql import SparkSession

from pyspark.sql import functions as F

from pyspark.sql import types

In [59]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [57]:
pyspark.__version__

'3.5.5'

In [4]:
!wget "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet"

--2025-03-02 12:45:19--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.239.238.133, 18.239.238.119, 18.239.238.152, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.239.238.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 64346071 (61M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2024-10.parquet’

yellow_tripdata_202  17%[==>                 ]  10.98M   342KB/s    in 8m 15s  

2025-03-02 12:53:47 (22.7 KB/s) - Connection closed at byte 11515640. Retrying.

--2025-03-02 12:53:48--  (try: 2)  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.239.238.133|:443... connected.
HTTP request sent, awaiting response... 206 Partial Content
Length: 64346071 (61M), 52830431 (50M) remaining [binary/octet-stream]
Sa

In [5]:
df = spark.read \
    .option("header", "true") \
    .parquet('yellow_tripdata_2024-10.parquet')

In [6]:
df.schema

StructType([StructField('VendorID', IntegerType(), True), StructField('tpep_pickup_datetime', TimestampNTZType(), True), StructField('tpep_dropoff_datetime', TimestampNTZType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('Airport_fee', DoubleType(), True)])

In [16]:
# get 1000 rows

df_pandas = pd.read_parquet('yellow_tripdata_2024-10.parquet', engine='pyarrow', columns=None)[:1000]


In [17]:
df_pandas.dtypes

VendorID                          int32
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag               object
PULocationID                      int32
DOLocationID                      int32
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
Airport_fee                     float64
dtype: object

In [18]:
spark.createDataFrame(df_pandas).schema

StructType([StructField('VendorID', LongType(), True), StructField('tpep_pickup_datetime', TimestampType(), True), StructField('tpep_dropoff_datetime', TimestampType(), True), StructField('passenger_count', DoubleType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', DoubleType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('Airport_fee', DoubleType(), True)])

In [49]:
yellow_schema = types.StructType([
    types.StructField("VendorID", types.IntegerType(), True),
    types.StructField("tpep_pickup_datetime", types.TimestampType(), True),
    types.StructField("tpep_dropoff_datetime", types.TimestampType(), True),
    types.StructField("passenger_count", types.LongType(), True),
    types.StructField("trip_distance", types.DoubleType(), True),
    types.StructField("RatecodeID", types.LongType(), True),
    types.StructField("store_and_fwd_flag", types.StringType(), True),
    types.StructField("PULocationID", types.IntegerType(), True),
    types.StructField("DOLocationID", types.IntegerType(), True),
    types.StructField("payment_type", types.LongType(), True),
    types.StructField("fare_amount", types.DoubleType(), True),
    types.StructField("extra", types.DoubleType(), True),
    types.StructField("mta_tax", types.DoubleType(), True),
    types.StructField("tip_amount", types.DoubleType(), True),
    types.StructField("tolls_amount", types.DoubleType(), True),
    types.StructField("improvement_surcharge", types.DoubleType(), True),
    types.StructField("total_amount", types.DoubleType(), True),
    types.StructField("congestion_surcharge", types.DoubleType(), True),
    types.StructField("Airport_fee", types.DoubleType(), True)
])

input_path = 'yellow_tripdata_2024-10.parquet'

df_yellow = spark.read \
    .option("header", "true") \
    .schema(yellow_schema) \
    .parquet(input_path)


output_path = f'data/pq/yellow/2024/10/'

df_yellow \
    .repartition(4) \
    .write.parquet(output_path)


In [60]:
df_yellow = spark.read.parquet('data/pq/yellow/*/*')

In [62]:
df_yellow.registerTempTable('trips_data_yellow')

/home/kabiromohd/spark/spark-3.5.5-bin-hadoop3/python/pyspark/sql/dataframe.py:329: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [66]:
spark.sql("""
SELECT COUNT(*)
FROM
    trips_data_yellow
WHERE  
    DATE(tpep_pickup_datetime) = "2024-10-15"
""").show()

+--------+
|count(1)|
+--------+
|  128097|
+--------+



In [68]:
spark.sql("""
SELECT MAX((unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600) AS longest_trip_hours
FROM trips_data_yellow
""").show()

+------------------+
|longest_trip_hours|
+------------------+
|162.61777777777777|
+------------------+



In [69]:
df_zones = spark.read.parquet('zones')

In [70]:
df_zones.columns

['locationid', 'borough', 'zone', 'service_zone']

In [71]:
df_zones.registerTempTable('zones')

In [73]:
spark.sql("""
SELECT z.zone, COUNT(t.PULocationID) AS pickup_count
FROM trips_data_yellow t
JOIN zones z ON t.PULocationID = z.Locationid
GROUP BY z.zone
ORDER BY pickup_count ASC
LIMIT 1;
""").show()

+--------------------+------------+
|                zone|pickup_count|
+--------------------+------------+
|Governor's Island...|           1|
+--------------------+------------+

